```python
1. pivot_wider: dr.pivot_wider()
   + basig usage
   + dr.pivot_wider(values_fill=0)
   + dr.pivot_wider(values_from=[many_cols], names_sep="_")
   + dr.pivot_wider(values_fn=...)
   
2. pivot_longer: dr.pivot_longer()
   + basig usage
   + dr.pivot_longer(cols=dr.starts_with())
   + dr.pivot_longer(names_sep=)
   + dr.pivot_longer(names_pattern=)
   + Combine dr.pivot_longer() with dr.pivot_wider()
   + Apply dr.pipe(lambda f: pd.wide_to_long(df=f))

3. crosstab, contigency/frequency table: dr.table()
```

In [15]:
import datar.all as dr
from datar import f
import polars as pl
from polars import col as c
import numpy as np
import warnings

warnings.filterwarnings("ignore")
pl.Config.set_tbl_cell_alignment("CENTER")

polars.config.Config

# <span style="color:#1E90FF">1. dr.pivot_wider()</span>

The dr.pivot_wider() converts unique values from one column into multiple columns in the DataFrame.   
=> Results in a wider DataFrame than the original.   

NOTE: if the chosen "index" column has duplicates => RAISE ERROR   

## 1.0. Create toy data

In [23]:
from datetime import date
dates = pl.date_range(date(2024, 1, 1), date(2024, 1, 31), eager=True) # 30 days
regions   = ['North', 'South', 'East', 'West']
products  = ['Widget', 'Gadget', 'Doohickey']

np.random.seed(42)
df_sales = dr.tibble(pl.DataFrame(
    {   'ID'        : range(1, 201),
        'date'      : np.random.choice(dates, size=200),
        'region'    : np.random.choice(regions, size=200),
        'product'   : np.random.choice(products, size=200),
        'quantity'  : np.random.randint(1, 20, size=200),
        'unit_price': np.round(np.random.uniform(5, 50, size=200), 2)
    }
    ).with_columns(sales = c("quantity") * c("unit_price")) # Add a column with the total sales amount
)

print(df_sales.head())

shape: (5, 7)
┌─────┬────────────┬────────┬───────────┬──────────┬────────────┬────────┐
│  ID ┆    date    ┆ region ┆  product  ┆ quantity ┆ unit_price ┆  sales │
│ --- ┆     ---    ┆   ---  ┆    ---    ┆    ---   ┆     ---    ┆   ---  │
│ i64 ┆    date    ┆   str  ┆    str    ┆    i64   ┆     f64    ┆   f64  │
╞═════╪════════════╪════════╪═══════════╪══════════╪════════════╪════════╡
│  1  ┆ 2024-01-07 ┆  West  ┆   Gadget  ┆    16    ┆    8.72    ┆ 139.52 │
│  2  ┆ 2024-01-20 ┆  North ┆   Gadget  ┆     7    ┆    44.45   ┆ 311.15 │
│  3  ┆ 2024-01-29 ┆  West  ┆ Doohickey ┆     4    ┆    29.82   ┆ 119.28 │
│  4  ┆ 2024-01-15 ┆  West  ┆   Gadget  ┆     1    ┆    12.42   ┆  12.42 │
│  5  ┆ 2024-01-11 ┆  South ┆   Widget  ┆     5    ┆    23.51   ┆ 117.55 │
└─────┴────────────┴────────┴───────────┴──────────┴────────────┴────────┘


## 1.1. dr.pivot_wider() basic usage

In [19]:
print(
    df_sales 
    >> dr.pivot_wider(
            names_from=f.region, 
            values_from=f.sales
        )
    >> dr.slice_head(n=5)
)

shape: (5, 9)
┌─────┬────────────┬───────────┬──────────┬───┬────────┬────────┬────────┬──────┐
│  ID ┆    date    ┆  product  ┆ quantity ┆ … ┆  West  ┆  North ┆  South ┆ East │
│ --- ┆     ---    ┆    ---    ┆    ---   ┆   ┆   ---  ┆   ---  ┆   ---  ┆  --- │
│ i64 ┆    date    ┆    str    ┆    i64   ┆   ┆   f64  ┆   f64  ┆   f64  ┆  f64 │
╞═════╪════════════╪═══════════╪══════════╪═══╪════════╪════════╪════════╪══════╡
│  1  ┆ 2024-01-07 ┆   Gadget  ┆    16    ┆ … ┆ 139.52 ┆  null  ┆  null  ┆ null │
│  2  ┆ 2024-01-20 ┆   Gadget  ┆     7    ┆ … ┆  null  ┆ 311.15 ┆  null  ┆ null │
│  3  ┆ 2024-01-29 ┆ Doohickey ┆     4    ┆ … ┆ 119.28 ┆  null  ┆  null  ┆ null │
│  4  ┆ 2024-01-15 ┆   Gadget  ┆     1    ┆ … ┆  12.42 ┆  null  ┆  null  ┆ null │
│  5  ┆ 2024-01-11 ┆   Widget  ┆     5    ┆ … ┆  null  ┆  null  ┆ 117.55 ┆ null │
└─────┴────────────┴───────────┴──────────┴───┴────────┴────────┴────────┴──────┘


## 1.2. dr.pivot_wider(values_fill=0)

In [20]:
print(
    df_sales  
    >> dr.pivot_wider(
            names_from=f.region, 
            values_from=f.sales,
            values_fill=0 # Fill NaN with 0
        )
    >> dr.slice_head(n=5)
)

shape: (5, 9)
┌─────┬────────────┬───────────┬──────────┬───┬────────┬────────┬────────┬──────┐
│  ID ┆    date    ┆  product  ┆ quantity ┆ … ┆  West  ┆  North ┆  South ┆ East │
│ --- ┆     ---    ┆    ---    ┆    ---   ┆   ┆   ---  ┆   ---  ┆   ---  ┆  --- │
│ i64 ┆    date    ┆    str    ┆    i64   ┆   ┆   f64  ┆   f64  ┆   f64  ┆  f64 │
╞═════╪════════════╪═══════════╪══════════╪═══╪════════╪════════╪════════╪══════╡
│  1  ┆ 2024-01-07 ┆   Gadget  ┆    16    ┆ … ┆ 139.52 ┆   0.0  ┆   0.0  ┆  0.0 │
│  2  ┆ 2024-01-20 ┆   Gadget  ┆     7    ┆ … ┆   0.0  ┆ 311.15 ┆   0.0  ┆  0.0 │
│  3  ┆ 2024-01-29 ┆ Doohickey ┆     4    ┆ … ┆ 119.28 ┆   0.0  ┆   0.0  ┆  0.0 │
│  4  ┆ 2024-01-15 ┆   Gadget  ┆     1    ┆ … ┆  12.42 ┆   0.0  ┆   0.0  ┆  0.0 │
│  5  ┆ 2024-01-11 ┆   Widget  ┆     5    ┆ … ┆   0.0  ┆   0.0  ┆ 117.55 ┆  0.0 │
└─────┴────────────┴───────────┴──────────┴───┴────────┴────────┴────────┴──────┘


## 1.3. dr.pivot_wider(values_from=[many_cols], names_sep="_")

In [21]:
print(
    df_sales  
    >> dr.pivot_wider(
            names_from=f.region,
            values_from=[f.sales, f.quantity], # Pivot multiple columns
            names_sep="_", # Separator between the new column names
            values_fill=0 # Fill NaN with 0
        )
    >> dr.slice_head(n=5)
)

shape: (5, 9)
┌─────┬────────────┬───────────┬──────────┬───┬────────┬────────┬────────┬──────┐
│  ID ┆    date    ┆  product  ┆ quantity ┆ … ┆  West  ┆  North ┆  South ┆ East │
│ --- ┆     ---    ┆    ---    ┆    ---   ┆   ┆   ---  ┆   ---  ┆   ---  ┆  --- │
│ i64 ┆    date    ┆    str    ┆    i64   ┆   ┆   f64  ┆   f64  ┆   f64  ┆  f64 │
╞═════╪════════════╪═══════════╪══════════╪═══╪════════╪════════╪════════╪══════╡
│  1  ┆ 2024-01-07 ┆   Gadget  ┆    16    ┆ … ┆ 139.52 ┆   0.0  ┆   0.0  ┆  0.0 │
│  2  ┆ 2024-01-20 ┆   Gadget  ┆     7    ┆ … ┆   0.0  ┆ 311.15 ┆   0.0  ┆  0.0 │
│  3  ┆ 2024-01-29 ┆ Doohickey ┆     4    ┆ … ┆ 119.28 ┆   0.0  ┆   0.0  ┆  0.0 │
│  4  ┆ 2024-01-15 ┆   Gadget  ┆     1    ┆ … ┆  12.42 ┆   0.0  ┆   0.0  ┆  0.0 │
│  5  ┆ 2024-01-11 ┆   Widget  ┆     5    ┆ … ┆   0.0  ┆   0.0  ┆ 117.55 ┆  0.0 │
└─────┴────────────┴───────────┴──────────┴───┴────────┴────────┴────────┴──────┘


## 1.4. dr.pivot_wider(values_fn=...)
values_fn: function to aggregate values   
           if there are multiple entries for the same index/column combination.   
           (e.g., sum, mean, len, etc.)   

In [22]:
print(
    df_sales
    >> dr.select(f.product, f.region, f.sales) # Select only relevant columns
    >> dr.pivot_wider(
            names_from=f.product, 
            values_from=f.sales,
            values_fill=0, # Fill NaN with 0
            values_fn=np.mean # Aggregate by sum if there are duplicates
        )
)

shape: (4, 4)
┌────────┬────────────┬────────────┬────────────┐
│ region ┆   Gadget   ┆  Doohickey ┆   Widget   │
│   ---  ┆     ---    ┆     ---    ┆     ---    │
│   str  ┆     f64    ┆     f64    ┆     f64    │
╞════════╪════════════╪════════════╪════════════╡
│  West  ┆ 199.315909 ┆ 300.025882 ┆  302.80913 │
│  North ┆ 293.631429 ┆ 250.478571 ┆ 287.189286 │
│  South ┆ 241.543333 ┆ 225.358333 ┆ 181.709375 │
│  East  ┆  396.0075  ┆   262.325  ┆ 286.317333 │
└────────┴────────────┴────────────┴────────────┘


# <span style="color:#1E90FF">2. dr.pivot_longer()</span>

## 2.0. Create toy data

In [24]:
n_patients = 8
patient_ids = [f"P{i:03d}" for i in range(1, n_patients+1)]

np.random.seed(42)
df_measurements = dr.tibble(pl.DataFrame({
    'patient_id' : patient_ids,
    'age'        : np.random.randint(20, 80, size=n_patients),
    # Day‑specific columns (wide format)
    'BP_day1'    : np.random.randint(110, 150, size=n_patients), # BP = Blood Pressure
    'HR_day1'    : np.random.randint(60, 100, size=n_patients), # HR = Heart Rate
    'BP_day2'    : np.random.randint(110, 150, size=n_patients),
    'HR_day2'    : np.random.randint(60, 100, size=n_patients),
    'BP_day3'    : np.random.randint(110, 150, size=n_patients),
    'HR_day3'    : np.random.randint(60, 100, size=n_patients)
}))

print(df_measurements)

shape: (8, 8)
┌────────────┬─────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│ patient_id ┆ age ┆ BP_day1 ┆ HR_day1 ┆ BP_day2 ┆ HR_day2 ┆ BP_day3 ┆ HR_day3 │
│     ---    ┆ --- ┆   ---   ┆   ---   ┆   ---   ┆   ---   ┆   ---   ┆   ---   │
│     str    ┆ i64 ┆   i64   ┆   i64   ┆   i64   ┆   i64   ┆   i64   ┆   i64   │
╞════════════╪═════╪═════════╪═════════╪═════════╪═════════╪═════════╪═════════╡
│    P001    ┆  58 ┆   128   ┆    62   ┆   142   ┆    62   ┆   134   ┆    67   │
│    P002    ┆  71 ┆   132   ┆    81   ┆   121   ┆    96   ┆   123   ┆    94   │
│    P003    ┆  48 ┆   120   ┆    61   ┆   131   ┆    66   ┆   118   ┆    73   │
│    P004    ┆  34 ┆   120   ┆    83   ┆   134   ┆    80   ┆   135   ┆    76   │
│    P005    ┆  62 ┆   133   ┆    89   ┆   136   ┆    68   ┆   111   ┆    95   │
│    P006    ┆  27 ┆   145   ┆    97   ┆   137   ┆    98   ┆   129   ┆    99   │
│    P007    ┆  40 ┆   149   ┆    61   ┆   125   ┆    77   ┆   137   ┆    63   │
│    P008    ┆

## 2.1. dr.pivot_longer() basic usage

In [32]:
print(
    df_measurements 
    >> dr.pivot_longer(
            cols=f[f.BP_day1 : f.HR_day3], # Specify columns to pivot
            names_to='measurement_day', # New column names
            values_to='value', # Name of the new value column,
            names_dtypes="categorical"
        )
    >> dr.slice_head(n=10)
)

shape: (10, 4)
┌────────────┬─────┬─────────────────┬───────┐
│ patient_id ┆ age ┆ measurement_day ┆ value │
│     ---    ┆ --- ┆       ---       ┆  ---  │
│     str    ┆ i64 ┆       enum      ┆  i64  │
╞════════════╪═════╪═════════════════╪═══════╡
│    P001    ┆  58 ┆     BP_day1     ┆  128  │
│    P001    ┆  58 ┆     HR_day1     ┆   62  │
│    P001    ┆  58 ┆     BP_day2     ┆  142  │
│    P001    ┆  58 ┆     HR_day2     ┆   62  │
│    P001    ┆  58 ┆     BP_day3     ┆  134  │
│    P001    ┆  58 ┆     HR_day3     ┆   67  │
│    P002    ┆  71 ┆     BP_day1     ┆  132  │
│    P002    ┆  71 ┆     HR_day1     ┆   81  │
│    P002    ┆  71 ┆     BP_day2     ┆  121  │
│    P002    ┆  71 ┆     HR_day2     ┆   96  │
└────────────┴─────┴─────────────────┴───────┘


## 2.2. dr.pivot_longer(cols=dr.starts_with())

In [30]:
print(
    df_measurements
    >> dr.pivot_longer(
            cols=dr.starts_with(match=["BP", "HR"]), # Specify columns to pivot
            names_to='measurement_day', # New column names
            values_to='value' # Name of the new value column
        )
    >> dr.slice_head(n=10)
)

shape: (10, 4)
┌────────────┬─────┬─────────────────┬───────┐
│ patient_id ┆ age ┆ measurement_day ┆ value │
│     ---    ┆ --- ┆       ---       ┆  ---  │
│     str    ┆ i64 ┆       enum      ┆  i64  │
╞════════════╪═════╪═════════════════╪═══════╡
│    P001    ┆  58 ┆     BP_day1     ┆  128  │
│    P001    ┆  58 ┆     BP_day2     ┆  142  │
│    P001    ┆  58 ┆     BP_day3     ┆  134  │
│    P001    ┆  58 ┆     HR_day1     ┆   62  │
│    P001    ┆  58 ┆     HR_day2     ┆   62  │
│    P001    ┆  58 ┆     HR_day3     ┆   67  │
│    P002    ┆  71 ┆     BP_day1     ┆  132  │
│    P002    ┆  71 ┆     BP_day2     ┆  121  │
│    P002    ┆  71 ┆     BP_day3     ┆  123  │
│    P002    ┆  71 ┆     HR_day1     ┆   81  │
└────────────┴─────┴─────────────────┴───────┘


## 2.3. dr.pivot_longer(names_sep=)

In [ ]:
print(
    df_measurements
    >> dr.pivot_longer(
            cols=dr.starts_with(match=["BP", "HR"]), # Specify columns to pivot
            names_sep="_", # Separator between the new column names
            names_to=['measurement', 'day'], # New column names
            values_to='value', # Name of the new value column
        )
    >> dr.slice_head(n=10)
)

## 2.4. dr.pivot_longer(names_pattern=)

In [34]:
print(
    df_measurements
    >> dr.pivot_longer(
            cols=dr.starts_with(match=["BP", "HR"]), # Specify columns to pivot
            names_to=['measurement', 'day'], # New column names
            names_pattern=r"([A-Z]+)_(day\d+)", # Regex pattern to extract new column names
            values_to='value' # Name of the new value column
        )
    >> dr.slice_head(n=10)
)

shape: (10, 4)
┌────────────┬─────┬─────────────┬───────┐
│ patient_id ┆ age ┆ measurement ┆ value │
│     ---    ┆ --- ┆     ---     ┆  ---  │
│     str    ┆ i64 ┆     enum    ┆  i64  │
╞════════════╪═════╪═════════════╪═══════╡
│    P001    ┆  58 ┆   BP_day1   ┆  128  │
│    P001    ┆  58 ┆   BP_day2   ┆  142  │
│    P001    ┆  58 ┆   BP_day3   ┆  134  │
│    P001    ┆  58 ┆   HR_day1   ┆   62  │
│    P001    ┆  58 ┆   HR_day2   ┆   62  │
│    P001    ┆  58 ┆   HR_day3   ┆   67  │
│    P002    ┆  71 ┆   BP_day1   ┆  132  │
│    P002    ┆  71 ┆   BP_day2   ┆  121  │
│    P002    ┆  71 ┆   BP_day3   ┆  123  │
│    P002    ┆  71 ┆   HR_day1   ┆   81  │
└────────────┴─────┴─────────────┴───────┘


In [35]:
print(
    df_measurements
    >> dr.pivot_longer(
            cols=dr.starts_with(match=["BP", "HR"]), # Specify columns to pivot
            names_to=['measurement', 'day'], # New column names
            names_pattern=r"([A-Z]+)_day(\d+)", # Take the integer part of the day only
            values_to='value' # Name of the new value column
        )
    >> dr.slice_head(n=10)
)

shape: (10, 4)
┌────────────┬─────┬─────────────┬───────┐
│ patient_id ┆ age ┆ measurement ┆ value │
│     ---    ┆ --- ┆     ---     ┆  ---  │
│     str    ┆ i64 ┆     enum    ┆  i64  │
╞════════════╪═════╪═════════════╪═══════╡
│    P001    ┆  58 ┆   BP_day1   ┆  128  │
│    P001    ┆  58 ┆   BP_day2   ┆  142  │
│    P001    ┆  58 ┆   BP_day3   ┆  134  │
│    P001    ┆  58 ┆   HR_day1   ┆   62  │
│    P001    ┆  58 ┆   HR_day2   ┆   62  │
│    P001    ┆  58 ┆   HR_day3   ┆   67  │
│    P002    ┆  71 ┆   BP_day1   ┆  132  │
│    P002    ┆  71 ┆   BP_day2   ┆  121  │
│    P002    ┆  71 ┆   BP_day3   ┆  123  │
│    P002    ┆  71 ┆   HR_day1   ┆   81  │
└────────────┴─────┴─────────────┴───────┘


## 2.5. Combine dr.pivot_longer() with dr.pivot_wider()

In [ ]:
print(
    df_measurements
    >> dr.pivot_longer(
            cols=dr.starts_with(match=["BP", "HR"]), # Specify columns to pivot
            names_to=['measurement', 'day'], # New column names
            names_pattern=r"([A-Z]+)_day(\d+)", # Take the integer part of the day only
            values_to='value' # Name of the new value column
        )
    >> dr.pivot_wider(
            names_from=f.measurement,
            values_from=f.value
        )
    >> dr.slice_head(n=10)
)

## 2.6. Apply dr.pipe(lambda f: pl.unpivot(df=f))